<a href="https://colab.research.google.com/github/officiallong/BUS4-118S-build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Laptop_Refund_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laptop Refund Agent — Google Gemini + LangGraph

## 1. Install libraries

In [ ]:
%pip install -q -U 'langchain>=1,<2' 'langchain-google-genai>=4,<5' 'langgraph>=1,<2'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 2. Connect to Gemini

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from google import genai

# Get Gemini API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Connect to Google Gemini
client = genai.Client(api_key=api_key)

# Use the model recommended by Google's API
model_name = "gemini-3.8-flash"

# Set up the Gemini model
model = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=api_key,
    temperature=0
)

# Test Gemini connection
print("Trying Gemini model:", model_name)

response = model.invoke("Hello!")
print("Gemini response:", response.content)

print("Gemini connection successful!")

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

## 3. Refund tools

In [ ]:
from langchain_core.tools import tool

SAMPLE_RETURN_DAYS = 30

@tool
def check_refund_policy() -> str:
    """Look up the sample laptop store refund policy, return window, and refund procedure."""
    return (
        "SAMPLE policy: Returns within 30 days of delivery; laptop must be in good "
        "condition; proof of purchase required; refunds use the original payment "
        "method. Contact customer support to request a return."
    )

@tool
def check_refund_eligibility(days_since_delivery: int, good_condition: bool, has_proof_of_purchase: bool) -> str:
    """Check sample refund eligibility using days since delivery, laptop condition, and receipt. Never guess missing information."""
    if days_since_delivery < 0:
        return "Invalid days_since_delivery; ask the customer to clarify."
    problems = []
    if days_since_delivery > SAMPLE_RETURN_DAYS:
        problems.append("more than 30 days since delivery")
    if not good_condition:
        problems.append("laptop not in good condition")
    if not has_proof_of_purchase:
        problems.append("proof of purchase missing")
    if problems:
        return "Not eligible under SAMPLE policy: " + "; ".join(problems)
    return "Meets SAMPLE eligibility criteria. Contact support to request a return; no refund has been approved or processed."

print("Refund tools ready.")


## 4. Refund Agent

In [ ]:
from langchain.agents import create_agent

agent_tools = [check_refund_policy, check_refund_eligibility]
system_prompt = """You are a laptop store Refund Agent for a classroom demo.
All refund rules are SAMPLE policies, not a real store's official policy.
Use your refund tools to answer policy questions and check hypothetical eligibility.
Never invent policy terms or claim a refund was approved, issued, or processed.
Ask for missing eligibility details instead of guessing them.
Be clear, polite, and concise. Only handle refunds and returns.
"""
agent_graph = create_agent(model=model, tools=agent_tools, system_prompt=system_prompt)
print("Refund Agent created successfully!")


## 5. Example 1 — Eligible return

In [ ]:
inputs = {"messages": [("user", "Can I return my laptop 15 days after delivery if it's in good condition and I have my receipt?")]}
result = agent_graph.invoke(inputs)
print("Agent returned:", result["messages"][-1].content)
print("\nStep by Step execution:")
for message in result["messages"]:
    print(message.pretty_repr())


## 6. Example 2 — Past the return deadline

In [ ]:
result2 = agent_graph.invoke({"messages": [("user", "My laptop arrived 45 days ago. It is in good condition and I have the receipt. Can I return it?")]})
print("Agent returned:", result2["messages"][-1].content)


## 7. Example 3 — Missing details

In [ ]:
result3 = agent_graph.invoke({"messages": [("user", "Can I get a refund for my laptop?")]})
print("Agent returned:", result3["messages"][-1].content)


## 8. Three-turn conversation memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

refund_agent_with_memory = create_agent(
    model=model, tools=agent_tools, system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "refund-demo"}}
for question in [
    "My laptop arrived 12 days ago.",
    "It's in good condition, and I have my receipt.",
    "Can I return it under the sample policy?",
]:
    conversation = refund_agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]}, config=config
    )
    print("Customer:", question)
    print("Agent:", conversation["messages"][-1].content, "\n")
